# 04 — Filter false-exclusion rate

The article's signature metric. A hard metadata filter can drop effective recall to zero
without changing the standard retrieval metrics. The gold doc is excluded *before* ranking
starts, so Recall@k computed over survivors looks fine.


In [1]:
# First: reproduce the article's worked example exactly
from rag_evals.evaluation.filter_exclusion import rate_against_survivors

DOCS = [
    {"id": "d1", "tenant": "acme",   "locale": "en-US"},
    {"id": "d2", "tenant": "acme",   "locale": "en-GB"},
    {"id": "d3", "tenant": "globex", "locale": "en-US"},
    {"id": "d4", "tenant": "acme",   "locale": "en-US"},
    {"id": "d5", "tenant": "acme",   "locale": "de-DE"},
]

def survivors_for(predicate):
    return {d["id"] for d in DOCS if all(d.get(k) == v for k, v in predicate.items())}

queries = [
    {"qid": "q1", "gold_doc_ids": ["d2"], "filter_predicate": {"locale": "en-US"}},
    {"qid": "q2", "gold_doc_ids": ["d4"], "filter_predicate": {"tenant": "acme"}},
    {"qid": "q3", "gold_doc_ids": ["d3"], "filter_predicate": {"tenant": "acme"}},
    {"qid": "q4", "gold_doc_ids": ["d5"], "filter_predicate": {"locale": "de-DE"}},
]
result = rate_against_survivors(queries, survivors_for)
print(f"filter_false_exclusion_rate = {result.rate:.0%}")
print(f"excluded queries: {[r.qid for r in result.rows if r.gold_excluded]}")


filter_false_exclusion_rate = 50%
excluded queries: ['q1', 'q3']


Now run on the live golden set. 30% of rows have a deliberately corrupted predicate
(see `data/golden.py`); the harness should detect them.


In [2]:
import json
from rag_evals.config import settings
from rag_evals.index.qdrant_store import QdrantStore
from rag_evals.evaluation.filter_exclusion import rate_against_survivors

rows = [json.loads(l) for l in (settings.golden_dir / "filter_aware.jsonl").open()][:100]
store = QdrantStore()

result = rate_against_survivors(rows, lambda p: store.survivor_ids(p))
print(f"filter_false_exclusion_rate = {result.rate:.2%}  ({result.n_excluded}/{result.n_queries})")


filter_false_exclusion_rate = 28.00%  (28/100)


The "aha": with a *bad* predicate, standard Recall@10 over the survivor set looks plausible —
but the gold doc was already gone. Compare:


In [3]:
from rag_evals.retrieval.dense import DenseRetriever
from rag_evals.evaluation.retrieval import recall_at_k

# Reuse the store opened above — embedded Qdrant locks one client per folder.
dense = DenseRetriever(store=store)
recalls = []
for r in rows[:30]:
    hits = dense(r["query"], limit=10, predicates=r["filter_predicate"])
    ranked = [h.doc_id for h in hits]
    recalls.append(recall_at_k(ranked, r["gold_doc_ids"], 10))

print(f"Recall@10 over survivor set: {sum(recalls)/len(recalls):.2%}")
print("Without filter_false_exclusion_rate, this number hides the broken queries.")


Recall@10 over survivor set: 56.33%
Without filter_false_exclusion_rate, this number hides the broken queries.
